# 分组聚合：groupby 从入门到精通

> 这是「数据分析从入门到精通」系列的第 14 篇。按品类、按地区、按月份……分组统计是数据分析最核心的操作之一。这篇把 groupby 从基础到进阶一次讲透。

---

嗨，我是小荷。

如果让我选 Pandas 里最重要的一个函数，我会毫不犹豫地说：`groupby`。

想象一下，你有一份 10 万行的电商订单，老板问你："各品类的销售额分别是多少？不同城市级别的平均客单价怎样？每个月的退款率有变化吗？" 这类问题，本质上都是**按某个维度分组，然后对每组做统计**。

萧何当年统计粮草，肯定也是按郡分组、按物资类型分组——这是人类管理复杂数据的本能。`groupby` 就是 Pandas 里实现这个本能的利器。

---

## groupby 基础：先分组，再聚合

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200
df = pd.DataFrame({
    '订单ID': range(1, n+1),
    '品类': np.random.choice(['数码', '服装', '食品', '图书'], n),
    '城市': np.random.choice(['北京', '上海', '广州', '深圳'], n),
    '金额': np.random.uniform(50, 2000, n).round(2),
    '数量': np.random.randint(1, 5, n),
    '月份': np.random.choice([1, 2, 3, 4], n)
})

# 按品类分组，求总销售额
result = df.groupby('品类')['金额'].sum()
print(result)
# 品类
# 图书    12345.67
# 数码    25678.90
# 服装    18234.56
# 食品    15432.10
# Name: 金额, dtype: float64


品类
图书    49364.75
数码    44504.98
服装    55296.01
食品    57540.61
Name: 金额, dtype: float64


理解 groupby 的三步：
1. **Split**：按指定列拆分成多个组
2. **Apply**：对每组应用聚合函数
3. **Combine**：把结果合并回来

```
df.groupby('品类')['金额'].sum()
         ↑ 分组依据   ↑ 要统计的列  ↑ 聚合方式
```

---

## 常用聚合函数

In [ ]:
g = df.groupby('品类')['金额']

g.sum()    # 总和
g.mean()   # 均值
g.median() # 中位数
g.max()    # 最大值
g.min()    # 最小值
g.count()  # 非空数量
g.std()    # 标准差
g.var()    # 方差
g.nunique()# 唯一值数量


---

## agg()：一次算多个指标

In [2]:
# 对金额列同时计算多个指标
result = df.groupby('品类')['金额'].agg(['sum', 'mean', 'count', 'std'])
print(result.round(2))

# 自定义列名
result = df.groupby('品类')['金额'].agg(
    总销售额=('sum'),
    平均客单价=('mean'),
    订单数=('count')
)

# 对多列分别做不同统计
result = df.groupby('品类').agg(
    总销售额=('金额', 'sum'),
    平均数量=('数量', 'mean'),
    订单数=('订单ID', 'count')
)
print(result.round(2))


         sum     mean  count     std
品类                                  
图书  49364.75   914.16     54  515.16
数码  44504.98   967.50     46  573.80
服装  55296.01  1202.09     46  527.90
食品  57540.61  1065.57     54  632.57
        总销售额  平均数量  订单数
品类                     
图书  49364.75  2.63   54
数码  44504.98  2.30   46
服装  55296.01  2.33   46
食品  57540.61  2.54   54


---

## 多列分组

In [3]:
# 按品类+城市分组
result = df.groupby(['品类', '城市'])['金额'].sum()
print(result)

# 转成透视表形式（更好看）
pivot = result.unstack()
print(pivot.round(2))
#         北京      上海      广州      深圳
# 品类
# 图书    XXXX    XXXX    XXXX    XXXX
# 数码    XXXX    XXXX    XXXX    XXXX


品类  城市
图书  上海    14226.58
    北京    11170.05
    广州    11129.22
    深圳    12838.90
数码  上海     7975.12
    北京    14746.93
    广州     7663.23
    深圳    14119.70
服装  上海     8765.25
    北京    13967.88
    广州    11804.41
    深圳    20758.47
食品  上海    10599.83
    北京    15469.16
    广州    18606.59
    深圳    12865.03
Name: 金额, dtype: float64
城市        上海        北京        广州        深圳
品类                                        
图书  14226.58  11170.05  11129.22  12838.90
数码   7975.12  14746.93   7663.23  14119.70
服装   8765.25  13967.88  11804.41  20758.47
食品  10599.83  15469.16  18606.59  12865.03


---

## transform()：保留原始行数

普通 groupby 聚合后行数变少了（每组一行），而 `transform()` 的结果**和原 DataFrame 行数一样**，适合给每行加上"组级别"的信息：

In [4]:
# 给每个订单加上"该品类的平均客单价"列
df['品类均价'] = df.groupby('品类')['金额'].transform('mean')

# 计算每个订单的金额偏差（该订单 - 该品类均价）
df['偏离均价'] = df['金额'] - df['品类均价']

# 每个组内的排名
df['组内排名'] = df.groupby('品类')['金额'].rank(ascending=False)


In [5]:
df

,订单ID,品类,城市,金额,数量,月份,品类均价,偏离均价,组内排名
0,1,食品,广州,1301.96,1,2,1065.566852,236.393148,23.0
1,2,图书,深圳,214.07,2,3,914.162037,-700.092037,48.0
2,3,数码,广州,365.18,1,1,967.499565,-602.319565,34.0
3,4,食品,北京,1802.18,1,1,1065.566852,736.613148,7.0
4,5,食品,深圳,1232.54,2,1,1065.566852,166.973148,27.0
...,...,...,...,...,...,...,...,...,...
195,196,服装,深圳,1864.98,1,1,1202.087174,662.892826,6.0
196,197,服装,深圳,1723.90,1,2,1202.087174,521.812826,9.0
197,198,图书,上海,886.54,4,3,914.162037,-27.622037,27.0
198,199,数码,深圳,1514.20,4,3,967.499565,546.700435,9.0


---

## filter()：过滤整组

In [6]:
# 只保留订单数 >= 50 的品类（整组保留或整组丢弃）
df_filtered = df.groupby('品类').filter(lambda x: len(x) >= 50)
df_filtered

,订单ID,品类,城市,金额,数量,月份,品类均价,偏离均价,组内排名
0,1,食品,广州,1301.96,1,2,1065.566852,236.393148,23.0
1,2,图书,深圳,214.07,2,3,914.162037,-700.092037,48.0
3,4,食品,北京,1802.18,1,1,1065.566852,736.613148,7.0
4,5,食品,深圳,1232.54,2,1,1065.566852,166.973148,27.0
5,6,图书,北京,67.93,4,1,914.162037,-846.232037,54.0
...,...,...,...,...,...,...,...,...,...
191,192,食品,广州,854.60,4,3,1065.566852,-210.966852,33.0
193,194,食品,深圳,1564.01,4,4,1065.566852,498.443148,16.0
194,195,食品,广州,714.57,4,2,1065.566852,-350.996852,35.0
197,198,图书,上海,886.54,4,3,914.162037,-27.622037,27.0


---

## 实战：电商数据分组分析

In [7]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500
df = pd.DataFrame({
    '订单ID': range(1, n+1),
    '品类': np.random.choice(['数码', '服装', '食品', '图书'], n,
                             p=[0.3, 0.3, 0.25, 0.15]),
    '城市': np.random.choice(['北京', '上海', '广州', '深圳', '杭州'], n),
    '金额': np.random.lognormal(6.5, 0.8, n).round(2),
    '数量': np.random.randint(1, 6, n),
    '月份': np.random.choice(range(1, 13), n),
    '是否退款': np.random.choice([True, False], n, p=[0.08, 0.92])
})

print("=" * 50)
print("【1】各品类销售概况")
summary = df.groupby('品类').agg(
    订单数=('订单ID', 'count'),
    总销售额=('金额', 'sum'),
    平均客单价=('金额', 'mean'),
    退款率=('是否退款', 'mean')
).round(2)
summary['退款率'] = summary['退款率'].map('{:.1%}'.format)
print(summary.sort_values('总销售额', ascending=False))

print("\n【2】各城市各品类销售额（透视）")
city_cat = df.groupby(['城市', '品类'])['金额'].sum().unstack().round(0)
print(city_cat)

print("\n【3】月度销售趋势（总额）")
monthly = df.groupby('月份')['金额'].sum().round(0)
for month, sales in monthly.items():
    bar = '█' * int(sales / monthly.max() * 20)
    print(f"  {month:2d}月 {bar} {sales:,.0f}")

print("\n【4】找出高价值品类（客单价超过全局均值）")
global_avg = df['金额'].mean()
high_value = df.groupby('品类')['金额'].mean()
print(high_value[high_value > global_avg].sort_values(ascending=False))


【1】各品类销售概况
    订单数       总销售额    平均客单价    退款率
品类                                
数码  154  161326.01  1047.57   7.0%
服装  139  125781.27   904.90   8.0%
食品  124  121888.82   982.97   8.0%
图书   83   77080.76   928.68  11.0%

【2】各城市各品类销售额（透视）
品类       图书       数码       服装       食品
城市                                    
上海   9392.0  27554.0  34307.0  17398.0
北京  19227.0  30108.0  28807.0  23469.0
广州  30674.0  30273.0  14969.0  29199.0
杭州  11248.0  45902.0  29327.0  32043.0
深圳   6539.0  27489.0  18370.0  19780.0

【3】月度销售趋势（总额）
   1月 █████████████████ 44,388
   2月 ██████████████ 37,907
   3月 █████████████████ 45,582
   4月 █████████████ 33,933
   5月 ███████████████ 40,868
   6月 ████████████████ 42,771
   7月 ████████████ 31,829
   8月 ██████████████ 36,730
   9月 ████████████████████ 51,660
  10月 ███████████ 29,552
  11月 ████████████████ 41,894
  12月 ██████████████████ 48,963

【4】找出高价值品类（客单价超过全局均值）
品类
数码    1047.571494
食品     982.974355
Name: 金额, dtype: float64


---

## 一个常见的坑：reset_index()

groupby 聚合后，分组列会变成**多级索引**，，需用`reset_index`重置为普通列才能进行后续 merge、筛选或导出（否则键会丢失或变成隐藏列）。

In [11]:
result = df.groupby('品类')['金额'].sum()
print(type(result))  # pandas.core.series.Series
print(result.index)  # Index(['图书', '数码', '服装', '食品'], dtype='object', name='品类')
# 如果需要"品类"变回普通列，加 reset_index()
result_df = result.reset_index()
print(result_df)
#    品类        金额
# 0  图书  12345.67
# 1  数码  25678.90

<class 'pandas.Series'>
Index(['图书', '数码', '服装', '食品'], dtype='str', name='品类')
   品类         金额
0  图书   77080.76
1  数码  161326.01
2  服装  125781.27
3  食品  121888.82


---

## 本篇小结

| 方法 | 用途 |
|------|------|
| `groupby(col).sum()` | 分组求和 |
| `groupby(col).agg({...})` | 多指标聚合 |
| `groupby(col).transform(func)` | 保留行数的组内计算 |
| `groupby(col).filter(func)` | 按组条件过滤 |
| `.unstack()` | 将多级索引转成透视表 |
| `.reset_index()` | 将索引变回普通列 |

---

## 课后练习

用下面的数据，完成分析任务：

In [13]:
import pandas as pd, numpy as np
np.random.seed(2024)
df = pd.DataFrame({
    '销售员': np.random.choice(['张三','李四','王五','赵六'], 100),
    '区域': np.random.choice(['华北','华东','华南','西部'], 100),
    '产品': np.random.choice(['A','B','C'], 100),
    '销售额': np.random.uniform(1000, 50000, 100).round(0),
    '季度': np.random.choice([1,2,3,4], 100)
})

# 任务1：统计每个销售员的总销售额，按降序排列
# 任务2：计算各区域各产品的销售额（透视表形式）
# 任务3：给每条记录加上"该销售员的季度均值"列
# 任务4：找出每个区域销售额最高的产品


In [14]:
# ========== 任务1：统计每个销售员的总销售额，按降序排列 ==========
print("\n" + "=" * 50)
print("任务1：每个销售员的总销售额（降序）")
print("=" * 50)
sales_by_person = df.groupby('销售员')['销售额'].sum().sort_values(ascending=False)
print(sales_by_person)
print(f"\n销冠: {sales_by_person.index[0]}，总销售额: {sales_by_person.iloc[0]:,.0f}")

# ========== 任务2：计算各区域各产品的销售额（透视表形式） ==========
print("\n" + "=" * 50)
print("任务2：各区域各产品销售额透视表")
print("=" * 50)
pivot = pd.pivot_table(df, values='销售额', index='区域', columns='产品', aggfunc='sum', margins=True)
print(pivot)

# ========== 任务3：给每条记录加上"该销售员的季度均值"列 ==========
print("\n" + "=" * 50)
print("任务3：新增'该销售员的季度均值'列")
print("=" * 50)
df['销售员_季度均值'] = df.groupby(['销售员', '季度'])['销售额'].transform('mean')
print(df[['销售员', '季度', '销售额', '销售员_季度均值']].head(10))

# ========== 任务4：找出每个区域销售额最高的产品 ==========
print("\n" + "=" * 50)
print("任务4：每个区域销售额最高的产品")
print("=" * 50)
region_product_sales = df.groupby(['区域', '产品'])['销售额'].sum().reset_index()
idx = region_product_sales.groupby('区域')['销售额'].idxmax()
best_product_per_region = region_product_sales.loc[idx]
print(best_product_per_region)


任务1：每个销售员的总销售额（降序）
销售员
王五    716814.0
赵六    649606.0
张三    609400.0
李四    563852.0
Name: 销售额, dtype: float64

销冠: 王五，总销售额: 716,814

任务2：各区域各产品销售额透视表
产品          A         B         C        All
区域                                          
华东   230463.0  113386.0  139905.0   483754.0
华北   153221.0  228353.0  165932.0   547506.0
华南   126420.0  342348.0  109021.0   577789.0
西部   324829.0  271611.0  334183.0   930623.0
All  834933.0  955698.0  749041.0  2539672.0

任务3：新增'该销售员的季度均值'列
  销售员  季度      销售额      销售员_季度均值
0  张三   1  20166.0  23473.142857
1  王五   4   5778.0  26558.600000
2  张三   2   3356.0  22105.666667
3  张三   1  17904.0  23473.142857
4  赵六   1  31910.0  21840.600000
5  张三   1  42042.0  23473.142857
6  王五   3  23988.0  21891.875000
7  赵六   4  18749.0  27475.166667
8  李四   4   5876.0  18268.166667
9  赵六   3  11521.0  23449.800000

任务4：每个区域销售额最高的产品
    区域 产品       销售额
0   华东  A  230463.0
4   华北  B  228353.0
7   华南  B  342348.0
11  西部  C  334183.0


评论区见 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 15 篇：多表合并 — merge / join / concat**
>
> 实际工作里数据往往分散在多张表——订单表、用户表、商品表……怎么把它们合在一起？SQL 里的 JOIN 在 Pandas 里怎么实现？下篇带你搞定。

---

👇 点「在看」，推给需要做分组统计的人  
💬 评论区说说你平时 groupby 最常用的场景是什么  
⭐ 关注公众号，跟萧何迷妹一起进阶

---

*「数据分析从入门到精通」系列 · 第 14 篇*  
*上一篇：[第 13 篇：字符串处理与正则表达式]*  
*下一篇：第 15 篇：多表合并 — merge / join / concat*